In [17]:
# 导入必要的库
import pandas as pd
import numpy as np
import requests
import time
from datetime import datetime
import pytz
from concurrent.futures import ThreadPoolExecutor, as_completed

# 设置pandas显示选项
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# 定义常量
BINANCE_SPOT_LIMIT = 1000
BINANCE_FUTURE_LIMIT = 1500
LOCAL_TZ = pytz.timezone('Asia/Shanghai')  # 使用上海时区，您可以根据需要更改

# 辅助函数
def generate_datetime(timestamp: float) -> datetime:
    """将时间戳转换为datetime对象"""
    dt = datetime.fromtimestamp(timestamp / 1000)
    return LOCAL_TZ.localize(dt)

def get_binance_data(symbol: str, exchange: str, start_time: str, end_time: str):
    """
    从Binance获取数据
    :param symbol: 交易对，如 'BTCUSDT'
    :param exchange: 'spot' 或 'future'
    :param start_time: 开始时间，格式：'YYYY-MM-DD'
    :param end_time: 结束时间，格式：'YYYY-MM-DD'
    :return: 包含K线数据的DataFrame
    """
    if exchange == 'spot':
        base_url = 'https://api.binance.com/api/v3/klines'
        limit = BINANCE_SPOT_LIMIT
    elif exchange == 'future':
        base_url = 'https://fapi.binance.com/fapi/v1/klines'
        limit = BINANCE_FUTURE_LIMIT
    else:
        raise ValueError("exchange must be either 'spot' or 'future'")

    start_time = int(datetime.strptime(start_time, '%Y-%m-%d').timestamp() * 1000)
    end_time = int(datetime.strptime(end_time, '%Y-%m-%d').timestamp() * 1000)

    all_data = []

    while start_time < end_time:
        url = f"{base_url}?symbol={symbol}&interval=1m&limit={limit}&startTime={start_time}"
        response = requests.get(url)
        data = response.json()

        if not data:
            break

        df = pd.DataFrame(data, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume', 
                                         'close_time', 'quote_asset_volume', 'number_of_trades', 
                                         'taker_buy_base_asset_volume', 'taker_buy_quote_asset_volume', 'ignore'])
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
        df = df.astype({'open': float, 'high': float, 'low': float, 'close': float, 'volume': float})
        all_data.append(df)

        start_time = int(df.iloc[-1]['close_time']) + 1

        print(f"Downloaded data until {df.iloc[-1]['timestamp']}")
        time.sleep(1)  # 为了避免触发频率限制

    return pd.concat(all_data, ignore_index=True)

# 主函数
def download_and_save(symbol, exchange, start_date, end_date, output_file):
    """
    下载数据并保存为pickle文件
    """
    print(f"开始下载 {symbol} {exchange} 数据从 {start_date} 到 {end_date}")
    df = get_binance_data(symbol, exchange, start_date, end_date)
    df.to_pickle(output_file)
    print(f"数据已保存到 {output_file}")
    return df

# 使用示例
# symbol = "BTCUSDT"
# exchange = "future"  # 或 "spot"
# start_date = "2016-01-01"
# end_date = "2024-10-21"
# output_file = f"{symbol}_{exchange}_{start_date}_{end_date}.pkl"
# 
# df = download_and_save(symbol, exchange, start_date, end_date, output_file)
# 
# # 显示数据概览
# print(df.head())
# print(df.info())

# 如果需要下载多个时间段或多个交易对，可以使用以下代码

def download_multiple(tasks):
    with ThreadPoolExecutor(max_workers=96) as executor:
        futures = [executor.submit(download_and_save, *task) for task in tasks]
        for future in as_completed(futures):
            try:
                future.result()
            except Exception as e:
                print(f"下载任务出错: {e}")

# 定义多个下载任务DOGE SOL XRP ada，avax，link，bch，dot，near，sui，uni，ltc
tasks = [
    ("ADAUSDT", "spot", "2017-01-01", "2024-10-21", "ADAUSDT_spot_2024.pkl"),
    # ("BTCUSDT", "future", "2022-07-01", "2022-12-31", "BTCUSDT_future_2022H2.pkl"),
    ("AVAXUSDT", "spot", "2017-01-01", "2024-10-21", "AVAXUSDT_spot_2024.pkl"),
    ("LINKUSDT", "spot", "2017-01-01", "2024-10-21", "LINKUSDT_spot_2024.pkl"),
    ("BCHUSDT", "spot", "2017-01-01", "2024-10-21", "BCHUSDT_spot_2024.pkl"),
    ("DOTUSDT", "spot", "2017-01-01", "2024-10-21", "DOTUSDT_spot_2024.pkl"),
    ("NEARUSDT", "spot", "2017-01-01", "2024-10-21", "NEARUSDT_spot_2024.pkl"),
    ("SUIUSDT", "spot", "2017-01-01", "2024-10-21", "SUIUSDT_spot_2024.pkl"),
    ("UNIUSDT", "spot", "2017-01-01", "2024-10-21", "UNIUSDT_spot_2024.pkl"),
    ("LTCUSDT", "spot", "2017-01-01", "2024-10-21", "LTCUSDT_spot_2024.pkl"),
]

# 执行多个下载任务
download_multiple(tasks)

开始下载 ADAUSDT spot 数据从 2017-01-01 到 2024-10-21
开始下载 AVAXUSDT spot 数据从 2017-01-01 到 2024-10-21
开始下载 LINKUSDT spot 数据从 2017-01-01 到 2024-10-21
开始下载 BCHUSDT spot 数据从 2017-01-01 到 2024-10-21
开始下载 DOTUSDT spot 数据从 2017-01-01 到 2024-10-21
开始下载 NEARUSDT spot 数据从 2017-01-01 到 2024-10-21
开始下载 SUIUSDT spot 数据从 2017-01-01 到 2024-10-21
开始下载 UNIUSDT spot 数据从 2017-01-01 到 2024-10-21
开始下载 LTCUSDT spot 数据从 2017-01-01 到 2024-10-21
Downloaded data until 2018-04-17 20:41:00
Downloaded data until 2019-11-29 02:39:00
Downloaded data until 2019-01-17 02:39:00
Downloaded data until 2020-09-22 23:09:00
Downloaded data until 2023-05-04 04:39:00
Downloaded data until 2020-09-17 19:39:00
Downloaded data until 2017-12-13 20:11:00
Downloaded data until 2020-10-14 21:39:00
Downloaded data until 2020-08-19 15:39:00
Downloaded data until 2018-04-18 13:21:00
Downloaded data until 2019-01-17 19:19:00
Downloaded data until 2019-11-29 19:19:00
Downloaded data until 2023-05-04 21:19:00
Downloaded data until 2020-08-20 08:1

In [18]:
df = pd.read_pickle("DOGEUSDT_spot_2024.pkl")
df.head(200)

,timestamp,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,taker_buy_quote_asset_volume,ignore
0,2019-07-05 12:00:00,0.004490,0.004600,0.003760,0.004200,60726008.0,1562328059999,259378.00423830,521,40516981.00000000,175346.09918580,0
1,2019-07-05 12:01:00,0.004200,0.004387,0.004200,0.004300,84307704.0,1562328119999,363010.82042050,561,40173084.00000000,173854.88612860,0
2,2019-07-05 12:02:00,0.004300,0.004475,0.004300,0.004475,48182744.0,1562328179999,210231.77591450,291,33036098.00000000,144634.35369460,0
3,2019-07-05 12:03:00,0.004450,0.004450,0.004169,0.004250,66457853.0,1562328239999,285857.80182680,289,8266236.00000000,35932.69492340,0
4,2019-07-05 12:04:00,0.004250,0.004385,0.004250,0.004350,22016425.0,1562328299999,95310.61966460,179,9954068.00000000,43020.45154170,0
...,...,...,...,...,...,...,...,...,...,...,...,...
195,2019-07-05 15:15:00,0.003811,0.003817,0.003810,0.003812,1759550.0,1562339759999,6705.84347670,26,53101.00000000,202.54458450,0
196,2019-07-05 15:16:00,0.003804,0.003812,0.003804,0.003804,571020.0,1562339819999,2172.08119780,10,16260.00000000,61.98637200,0
197,2019-07-05 15:17:00,0.003804,0.003812,0.003804,0.003811,166110.0,1562339879999,632.44729060,6,71245.00000000,271.58238400,0
198,2019-07-05 15:18:00,0.003811,0.003811,0.003800,0.003800,640544.0,1562339939999,2437.21890270,14,21087.00000000,80.36206100,0


In [ ]:
from tardis_client import TardisClient
import asyncio

async def get_minute_data():
    tardis_client = TardisClient(api_key="YOUR_API_KEY")
    messages = tardis_client.replay(
        exchange="binance",
        from_date="2022-01-01T00:00:00",
        to_date="2022-01-01T00:01:00",
        filters=[{"name": "trades", "symbols": ["BTCUSDT"]}]
    )
    async for local_timestamp, message in messages:
        print(message)

asyncio.run(get_minute_data())
